# Parametric SELE Posterior Training

Trains a network that reads an ELE measurement and returns a Gaussian over the five
physical simulator parameters (`p0`, `D`, `S`, `tau`, `alpha_scale`), including the full
covariance between them.

**Before running, put these three files in one Google Drive folder:**

1. `parametric_100k_train.npz` — from `Data/parametric_model/datasets/`
2. `model_definition.py` — from `src/regularization/parametric_model/`
3. `features.py` — from `src/regularization/parametric_model/`

Then set `DRIVE_DIR` in the config cell.

The network is small, so a GPU is convenient but not required. Checkpoints and the final
`.pt` are written back to the same Drive folder.

**Watch the coverage numbers, not just the loss.** The likelihood can be driven down by a
network that is confidently wrong. Coverage is the fraction of held-out truths that land
inside the predicted 68% and 95% ellipsoids, and it should sit near those percentages. Far
below means the error bars are too tight to believe; far above means they are uselessly wide.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU. This model is small enough that CPU is fine.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# -- CONFIGURATION ------------------------------------------------
DRIVE_DIR = '/content/drive/MyDrive/Thesis/Colab Model Training'  # <-- your folder
DATA_FILE = 'parametric_100k_train.npz'
OUTPUT_FILE = 'parametric_posterior.pt'
CHECKPOINT_EVERY = 25
# -----------------------------------------------------------------

import os

data_path = os.path.join(DRIVE_DIR, DATA_FILE)
output_path = os.path.join(DRIVE_DIR, OUTPUT_FILE)
resume_path = os.path.join(DRIVE_DIR, 'checkpoint_resume_parametric.pt')

for required in (DATA_FILE, 'model_definition.py', 'features.py'):
    assert os.path.exists(os.path.join(DRIVE_DIR, required)), f'missing {required} in {DRIVE_DIR}'
print('All inputs present in', DRIVE_DIR)

In [ ]:
import sys
import time
from dataclasses import dataclass, asdict, field
from typing import Tuple

import numpy as np
from scipy.stats import chi2
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

sys.path.insert(0, DRIVE_DIR)
import features as feat
from model_definition import build_parametric_network, gaussian_nll

print('Imports OK')

In [ ]:
@dataclass
class TrainingConfig:
    data_path: str = ''
    output_path: str = ''
    input_mode: str = 'log_amplitude_shape'
    hidden_dims: Tuple[int, ...] = (256, 256, 256)
    use_layer_norm: bool = True
    use_residual: bool = True
    batch_size: int = 512
    learning_rate: float = 1e-3
    num_epochs: int = 200
    weight_decay: float = 0.0
    validation_fraction: float = 0.1
    seed: int = 42
    input_dim: int = 0
    n_parameters: int = 5

config = TrainingConfig(data_path=data_path, output_path=output_path)
config

In [ ]:
torch.manual_seed(config.seed)
np.random.seed(config.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

payload = np.load(config.data_path, allow_pickle=False)
ele = payload['ele']
targets = payload['params_normalized']
print(f'{ele.shape[0]} curves, {ele.shape[1]} wavelengths')

amplitude_spread, shape_spread = feat.describe(ele)
print(f'amplitude spread {amplitude_spread:.2f} decades, shape spread {shape_spread:.2f} decades')

rng = np.random.default_rng(config.seed)
order = rng.permutation(ele.shape[0])
n_validation = int(round(config.validation_fraction * ele.shape[0]))
validation_idx, train_idx = order[:n_validation], order[n_validation:]

# Statistics from the training split only, so validation stays an honest estimate.
input_stats = feat.fit_input_stats(ele[train_idx], config.input_mode)
inputs = feat.transform(ele, config.input_mode, input_stats)
config.input_dim = inputs.shape[1]

to_tensor = lambda a: torch.tensor(a, dtype=torch.float32)
train_loader = DataLoader(
    TensorDataset(to_tensor(inputs[train_idx]), to_tensor(targets[train_idx])),
    batch_size=config.batch_size, shuffle=True, num_workers=2, pin_memory=True)
validation_loader = DataLoader(
    TensorDataset(to_tensor(inputs[validation_idx]), to_tensor(targets[validation_idx])),
    batch_size=4096, shuffle=False)

param_spec = {
    'names': [str(n) for n in payload['param_names']],
    'lower': [float(v) for v in payload['param_lower']],
    'upper': [float(v) for v in payload['param_upper']],
    'is_log': [bool(v) for v in payload['param_is_log']],
}
print('input_dim =', config.input_dim)

In [ ]:
network = build_parametric_network(asdict(config)).to(device)
print(f'{sum(p.numel() for p in network.parameters())} parameters')

optimizer = torch.optim.Adam(network.parameters(), lr=config.learning_rate,
                             weight_decay=config.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config.num_epochs, eta_min=config.learning_rate * 1e-2)

start_epoch = 0
if os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location=device)
    network.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch']
    print(f'Resuming from epoch {start_epoch}')

THRESHOLD_68 = chi2.ppf(0.6827, df=config.n_parameters)
THRESHOLD_95 = chi2.ppf(0.9545, df=config.n_parameters)

@torch.no_grad()
def evaluate():
    network.eval()
    total_nll, total_n, inside_68, inside_95 = 0.0, 0, 0, 0
    for x, y in validation_loader:
        x, y = x.to(device), y.to(device)
        mean, scale_tril = network(x)
        total_nll += gaussian_nll(mean, scale_tril, y).item() * x.shape[0]
        total_n += x.shape[0]
        whitened = torch.linalg.solve_triangular(
            scale_tril, (y - mean).unsqueeze(-1), upper=False).squeeze(-1)
        radius = (whitened ** 2).sum(dim=1)
        inside_68 += int((radius <= THRESHOLD_68).sum())
        inside_95 += int((radius <= THRESHOLD_95).sum())
    network.train()
    return total_nll / total_n, inside_68 / total_n, inside_95 / total_n

In [ ]:
# -- TRAINING LOOP ------------------------------------------------
start_time = time.time()

for epoch in tqdm(range(start_epoch, config.num_epochs), desc='Training',
                  initial=start_epoch, total=config.num_epochs):
    running, seen = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        mean, scale_tril = network(x)
        loss = gaussian_nll(mean, scale_tril, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
        optimizer.step()
        running += loss.item() * x.shape[0]
        seen += x.shape[0]
    scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == start_epoch:
        nll, cov68, cov95 = evaluate()
        print(f'Epoch {epoch+1}/{config.num_epochs} | train {running/seen:8.4f} | '
              f'val {nll:8.4f} | coverage {cov68*100:5.1f}% / {cov95*100:5.1f}% | '
              f'{time.time()-start_time:.0f}s')

    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        torch.save({'epoch': epoch + 1,
                    'model_state_dict': network.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict()}, resume_path)

nll, cov68, cov95 = evaluate()
print(f'\nFinal: val NLL {nll:.4f}, coverage {cov68*100:.1f}% / {cov95*100:.1f}%')

In [ ]:
# -- SAVE FINAL MODEL ---------------------------------------------
state = {k.replace('_orig_mod.', ''): v for k, v in network.state_dict().items()}
torch.save({
    'model_state_dict': state,
    'config': asdict(config),
    'input_stats': {'mean': np.asarray(input_stats['mean']),
                    'std': np.asarray(input_stats['std'])},
    'param_spec': param_spec,
}, config.output_path)

if os.path.exists(resume_path):
    os.remove(resume_path)

print('Saved', config.output_path)
print('Download it and place it in Data/parametric_model/models/ in the repo.')